# Healthcare Knowledge Graph - Vector Indexing and Embedding

This notebook demonstrates how to create vector embeddings for healthcare providers in Neo4j and perform semantic similarity searches.

## 1. Imports

In [ ]:
from dotenv import load_dotenv
import os
from langchain_community.graphs import Neo4jGraph
from langchain_openai import ChatOpenAI

## 2. Load Environment Variables

In [ ]:
load_dotenv()

AURA_INSTANCENAME = os.environ["AURA_INSTANCENAME"]
NEO4J_URI = os.environ["NEO4J_URI"]
NEO4J_USERNAME = os.environ["NEO4J_USERNAME"]
NEO4J_PASSWORD = os.environ["NEO4J_PASSWORD"]
NEO4J_DATABASE = os.environ["NEO4J_DATABASE"]
AUTH = (NEO4J_USERNAME, NEO4J_PASSWORD)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_ENDPOINT = os.getenv("OPENAI_ENDPOINT")

## 3. Initialize ChatOpenAI

In [ ]:
chat = ChatOpenAI(api_key=OPENAI_API_KEY)

## 4. Initialize Neo4j Graph Connection

In [ ]:
kg = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE,
)

## 5. Create Vector Index for Healthcare Providers

Creates a vector index on the `comprehensiveEmbedding` property of `HealthcareProvider` nodes.
- **Dimensions**: 1536 (OpenAI embedding size)
- **Similarity Function**: Cosine similarity

In [ ]:
kg.query(
    """
    CREATE VECTOR INDEX health_providers_embeddings IF NOT EXISTS
    FOR (hp:HealthcareProvider) ON (hp.comprehensiveEmbedding)
    OPTIONS {
      indexConfig: {
        `vector.dimensions`: 1536,
        `vector.similarity_function`: 'cosine'
      }
    }
    """
)

## 6. Verify Vector Index Creation

In [ ]:
res = kg.query(
    """
  SHOW VECTOR INDEXES
  """
)
print(res)

## 7. Generate and Store Embeddings for Healthcare Providers

This query:
1. Finds all HealthcareProvider nodes that treat patients and have a bio
2. Encodes the bio text using OpenAI's embedding API
3. Stores the embedding vector in the `comprehensiveEmbedding` property

In [ ]:
kg.query(
    """
    MATCH (hp:HealthcareProvider)-[:TREATS]->(p:Patient)
    WHERE hp.bio IS NOT NULL
    WITH hp, genai.vector.encode(
        hp.bio,
        "OpenAI",
        {
          token: $openAiApiKey,
          endpoint: $openAiEndpoint
        }) AS vector
    WITH hp, vector
    WHERE vector IS NOT NULL
    CALL db.create.setNodeVectorProperty(hp, "comprehensiveEmbedding", vector)
    """,
    params={
        "openAiApiKey": OPENAI_API_KEY,
        "openAiEndpoint": OPENAI_ENDPOINT,
    },
)

## 8. Verify Embeddings are Stored

In [ ]:
result = kg.query(
    """
    MATCH (hp:HealthcareProvider)
    WHERE hp.bio IS NOT NULL
    RETURN hp.bio, hp.name, hp.comprehensiveEmbedding
    LIMIT 5
    """
)

# Loop through the results
for record in result:
    print(f"Name: {record['hp.name']}")
    print(f"Bio: {record['hp.bio']}")
    print("---")

## 9. Semantic Similarity Search - Query Healthcare Providers

This performs a vector similarity search to find healthcare providers matching a natural language query.

In [ ]:
question = "give me a list of healthcare providers in the area of dermatology"

## 10. Execute Vector Similarity Search

In [ ]:
result = kg.query(
    """
    WITH genai.vector.encode(
        $question,
        "OpenAI",
        {
          token: $openAiApiKey,
          endpoint: $openAiEndpoint
        }) AS question_embedding
    CALL db.index.vector.queryNodes(
        'health_providers_embeddings',
        $top_k,
        question_embedding
        ) YIELD node AS healthcare_provider, score
    RETURN healthcare_provider.name, healthcare_provider.bio, score
    """,
    params={
        "openAiApiKey": OPENAI_API_KEY,
        "openAiEndpoint": OPENAI_ENDPOINT,
        "question": question,
        "top_k": 3,
    },
)

## 11. Display Search Results

In [ ]:
for record in result:
    print(f"Name: {record['healthcare_provider.name']}")
    print(f"Bio: {record['healthcare_provider.bio']}")
    print(f"Score: {record['score']}")
    print("---")